# 01 Ingest: CSV เป็น Parquet

แปลง CSV 13 ไฟล์ (~2.9 GB) เป็น Parquet แบ่งโฟลเดอร์ตามปี

ขั้นตอน
- อ่าน CSV โดยกำหนด type ของแต่ละคอลัมน์เอง ไม่ให้ DuckDB เดา
- ตรวจคุณภาพข้อมูลดิบ: ค่าว่าง, ช่วงของค่า, ชื่อสถานี, แถวซ้ำ
- ตัดแถวซ้ำ แล้วเขียน Parquet ทีละปี เรียงตาม date, hour, origin, dest
- เช็คว่า Parquet ตรงกับ CSV หลังตัดแถวซ้ำทุกแถว
- เช็คว่า row group เรียงตามวันที่จริง

In [1]:
import shutil
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import pandas as pd
import plotly.graph_objects as go
import pyarrow.parquet as pq

from bart import viz
from bart.config import OD, RAW, duck, fetch_row, fetch_value
from bart.perf import stage
from bart.stations import STATIONS

viz.notebook_mode()

## อ่าน CSV

กำหนด type เอง Trip Count อ่านเป็น UINTEGER ไว้ก่อน แล้วค่อยดูค่าสูงสุดว่าลดเหลือ USMALLINT (สูงสุด 65,535) ได้ไหม

`csv_with_file` เพิ่มคอลัมน์ file_year (ปีจากชื่อไฟล์) ไว้ใช้ตอนหาแถวซ้ำ

In [2]:
con = duck()

columns = {
    "Date": "DATE",
    "Hour": "UTINYINT",
    "Origin Station": "VARCHAR",
    "Destination Station": "VARCHAR",
    "Trip Count": "UINTEGER",
}

csv = f"read_csv('{RAW}/*.csv', header=true, columns={columns})"

csv_with_file = f"""
    (
        SELECT
            *,
            regexp_extract(filename, '([0-9]{{4}})[.]csv$', 1)::INTEGER AS file_year
        FROM read_csv('{RAW}/*.csv', header=true, columns={columns}, filename=true)
    )
"""

## ตรวจคุณภาพข้อมูลดิบ

ดูค่าว่าง, ช่วงของ hour และ trips, ช่วงวันที่ และจำนวนทริปที่เข้าและออกสถานีเดียวกัน

In [3]:
quality_query = f"""
    SELECT
        count(*)                                                AS rows,
        count(*) FILTER (WHERE "Date" IS NULL)                  AS null_date,
        count(*) FILTER (WHERE "Hour" IS NULL)                  AS null_hour,
        count(*) FILTER (WHERE "Origin Station" IS NULL)        AS null_origin,
        count(*) FILTER (WHERE "Destination Station" IS NULL)   AS null_dest,
        count(*) FILTER (WHERE "Trip Count" IS NULL)            AS null_trips,
        min("Hour")                                             AS min_hour,
        max("Hour")                                             AS max_hour,
        min("Trip Count")                                       AS min_trips,
        max("Trip Count")                                       AS max_trips,
        min("Date")                                             AS first_date,
        max("Date")                                             AS last_date,
        count(*) FILTER (WHERE "Origin Station" = "Destination Station")           AS same_station_rows,
        sum("Trip Count") FILTER (WHERE "Origin Station" = "Destination Station")  AS same_station_trips
    FROM {csv}
"""

quality = con.sql(quality_query).df().iloc[0]
quality.to_frame("value")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,value
rows,112536132
null_date,0
null_hour,0
null_origin,0
null_dest,0
null_trips,0
min_hour,0
max_hour,23
min_trips,1
max_trips,1826


In [4]:
null_columns = ["null_date", "null_hour", "null_origin", "null_dest", "null_trips"]
for column in null_columns:
    assert quality[column] == 0, f"{column} has {quality[column]} rows"

assert quality["min_hour"] == 0
assert quality["max_hour"] == 23
assert quality["min_trips"] >= 1
assert quality["max_trips"] < 65_535, "trips does not fit in USMALLINT"

print("no nulls, hour 0-23, trips between", quality["min_trips"], "and", quality["max_trips"])

no nulls, hour 0-23, trips between 1 and 1826


ไม่มีค่าว่าง hour อยู่ในช่วง 0-23 และ trips มีค่าตั้งแต่ 1 ขึ้นไป แปลว่า feed ไม่ได้เก็บแถวที่มี 0 ทริปไว้ ส่วนค่าสูงสุดของ trips พอดีกับ USMALLINT

มีทริปที่เข้าและออกสถานีเดียวกันด้วย ยังเก็บไว้ตามข้อมูลดิบก่อน ค่อยตัดสินใจตอนทำ marts ว่าจะเอาไปใช้แบบไหน

## ชื่อสถานี

ชื่อสถานีที่เจอใน CSV ต้องตรงกับ `src/bart/stations.py` ครบทั้ง 50 สถานี

In [5]:
station_query = f"""
    SELECT "Origin Station" AS station FROM {csv}
    UNION
    SELECT "Destination Station" AS station FROM {csv}
    ORDER BY station
"""

stations = []
for row in con.sql(station_query).fetchall():
    stations.append(row[0])

print(len(stations), "stations")
print(", ".join(stations))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

50 stations
12TH, 16TH, 19TH, 24TH, ANTC, ASHB, BALB, BAYF, BERY, CAST, CIVC, COLM, COLS, CONC, DALY, DBRK, DELN, DUBL, EMBR, FRMT, FTVL, GLEN, HAYW, LAFY, LAKE, MCAR, MLBR, MLPT, MONT, NBRK, NCON, OAKL, ORIN, PCTR, PHIL, PITT, PLZA, POWL, RICH, ROCK, SANL, SBRN, SFIA, SHAY, SSAN, UCTY, WARM, WCRK, WDUB, WOAK


In [6]:
not_in_stations_py = []
for station in stations:
    if station not in STATIONS:
        not_in_stations_py.append(station)

not_in_csv = []
for station in STATIONS:
    if station not in stations:
        not_in_csv.append(station)

print("in CSV but not in stations.py:", not_in_stations_py)
print("in stations.py but not in CSV:", not_in_csv)

assert len(not_in_stations_py) == 0
assert len(not_in_csv) == 0

in CSV but not in stations.py: []
in stations.py but not in CSV: []


## แถวซ้ำ

key ของข้อมูลคือ date + hour + origin + dest ควรมีได้แค่แถวเดียวต่อ key
ตรงนี้นับว่ามี key ไหนที่โผล่มากกว่า 1 ครั้ง และมาจากไฟล์ไหน

In [7]:
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE duplicates AS
    SELECT
        "Date"                         AS date,
        "Hour"                         AS hour,
        "Origin Station"               AS origin,
        "Destination Station"          AS dest,
        count(*)                       AS copies,
        count(DISTINCT "Trip Count")   AS distinct_trip_values,
        max("Trip Count")              AS trips,
        min(file_year)                 AS first_file,
        max(file_year)                 AS second_file
    FROM {csv_with_file}
    GROUP BY "Date", "Hour", "Origin Station", "Destination Station"
    HAVING count(*) > 1
""")

duplicate_keys = fetch_value(con, "SELECT count(*) FROM duplicates")
print("keys that appear more than once:", f"{duplicate_keys:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

keys that appear more than once: 16,078


In [8]:
duplicates_by_day = con.sql("""
    SELECT
        date,
        first_file,
        second_file,
        count(*)                   AS duplicate_rows,
        min(hour)                  AS first_hour,
        max(hour)                  AS last_hour,
        max(copies)                AS max_copies,
        max(distinct_trip_values)  AS max_distinct_trip_values,
        sum(trips * (copies - 1))  AS extra_trips
    FROM duplicates
    GROUP BY date, first_file, second_file
    ORDER BY date
""").df()

duplicates_by_day

,date,first_file,second_file,duplicate_rows,first_hour,last_hour,max_copies,max_distinct_trip_values,extra_trips
0,2012-01-01,2011,2012,2290,0,2,2,1,37643.0
1,2013-01-01,2012,2013,2407,0,2,2,1,38770.0
2,2014-01-01,2013,2014,2453,0,2,2,1,41534.0
3,2015-01-01,2014,2015,2464,0,2,2,1,37483.0
4,2016-01-01,2015,2016,2367,0,2,2,1,32040.0
5,2017-01-01,2016,2017,2080,0,2,2,1,29130.0
6,2018-01-01,2017,2018,2017,0,2,2,1,24383.0


In [9]:
extra_rows = fetch_value(con, "SELECT sum(copies - 1) FROM duplicates")
extra_trips = fetch_value(con, "SELECT sum(trips * (copies - 1)) FROM duplicates")
max_copies = fetch_value(con, "SELECT max(copies) FROM duplicates")
conflicting_keys = fetch_value(con, "SELECT count(*) FROM duplicates WHERE distinct_trip_values > 1")

print("extra rows      :", f"{extra_rows:,}")
print("extra trips     :", f"{extra_trips:,}")
print("max copies      :", max_copies)
print("conflicting keys:", conflicting_keys)

assert conflicting_keys == 0, "some duplicated keys have different trip counts"

extra rows      : 16,078
extra trips     : 240,983
max copies      : 2
conflicting keys: 0


แถวซ้ำทั้งหมดเป็นช่วง hour 0-2 ของวันที่ 1 มกราคม ปี 2012-2018 ที่ติดมาท้ายไฟล์ของปีก่อนหน้า และมีอยู่ในไฟล์ของปีนั้นเองด้วย
น่าจะเพราะคืนส่งท้ายปี BART เปิดบริการเลยเที่ยงคืน ทริปหลังเที่ยงคืนเลยถูกนับรวมในไฟล์ปีเก่าด้วย

ทุก key ที่ซ้ำมีค่า trips เท่ากันทุกตัว (conflicting keys = 0) จึงเป็นการซ้ำแบบเป๊ะๆ ตัดออกด้วย DISTINCT ได้เลย ไม่ต้องเลือกว่าจะเชื่อแถวไหน

ถ้าไม่ตัด ยอดช่วงตี 0-2 ของวันปีใหม่จะสูงเป็น 2 เท่า

## เขียน Parquet

ขั้นตอน
1. โหลด CSV เข้า temp table ครั้งเดียว ใช้ DISTINCT ตัดแถวซ้ำ และแปลง type ไปพร้อมกัน
2. เขียนทีละปี ปีละ 1 ไฟล์ เรียงตาม date, hour, origin, dest

ที่ไม่ใช้ `PARTITION_BY (year)` เพราะลองแล้ว DuckDB เขียนหลายไฟล์พร้อมกันโดยไม่รักษาลำดับ ถึงจะใส่ ORDER BY ข้อมูลก็ไม่เรียง
ส่วน `preserve_insertion_order` ต้องเปิดตอนเขียน ไม่อย่างนั้น ORDER BY จะไม่มีผลกับไฟล์ที่ออกมา

สถานีเก็บเป็น ENUM 50 ค่า ใน Parquet จะถูกเก็บแบบ dictionary ใช้พื้นที่น้อยมาก
ROW_GROUP_SIZE 1 ล้านแถว ประมาณ 1 เดือนต่อ row group

In [10]:
quoted_stations = []
for station in stations:
    quoted_stations.append(f"'{station}'")
station_list = ", ".join(quoted_stations)

con.execute("DROP TABLE IF EXISTS od")
con.execute("DROP TYPE IF EXISTS station")
con.execute(f"CREATE TYPE station AS ENUM ({station_list})")

print(len(stations), "stations in ENUM")

50 stations in ENUM


In [11]:
def write_year(con, year):
    folder = OD / f"year={year}"
    folder.mkdir(parents=True)

    con.execute(f"""
        COPY (
            SELECT date, hour, origin, dest, trips
            FROM od
            WHERE year(date) = {year}
            ORDER BY date, hour, origin, dest
        )
        TO '{folder / "data_0.parquet"}' (
            FORMAT parquet,
            COMPRESSION zstd,
            COMPRESSION_LEVEL 6,
            ROW_GROUP_SIZE 1000000
        )
    """)

In [12]:
if OD.exists():
    shutil.rmtree(OD)

load_query = f"""
    CREATE TEMP TABLE od AS
    SELECT DISTINCT
        "Date"                         AS date,
        "Hour"                         AS hour,
        "Origin Station"::station      AS origin,
        "Destination Station"::station AS dest,
        "Trip Count"::USMALLINT        AS trips
    FROM {csv}
"""

with stage("01.csv_to_parquet"):
    con.execute(load_query)

    years = []
    for row in con.sql("SELECT DISTINCT year(date) FROM od ORDER BY 1").fetchall():
        years.append(row[0])

    con.execute("SET preserve_insertion_order = true")
    for year in years:
        write_year(con, year)
    con.execute("SET preserve_insertion_order = false")

con.execute("DROP TABLE od")
print("years written:", years)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[perf] 01.csv_to_parquet: 9.16s | 12.24 cores | peak RAM 7.224 GB | CPU 75.0%
years written: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


## เช็คความถูกต้อง

เทียบ Parquet กับ CSV ทีละปี โดยฝั่ง CSV คำนวณใหม่จากไฟล์ดิบ (ไม่ใช้ temp table od) แล้วตัดแถวซ้ำด้วย DISTINCT

เช็ค 4 อย่าง
- จำนวนแถวและผลรวมทริปต่อปีตรงกัน
- ยอดรวมเท่ากับ CSV ลบแถวซ้ำที่นับได้ข้างบน
- ไม่มี key ซ้ำเหลือใน Parquet
- ทุกแถวอยู่ในโฟลเดอร์ปีที่ถูกต้อง

In [13]:
parquet = f"""
    read_parquet(
        '{OD}/*/*.parquet',
        hive_partitioning = true,
        hive_types = {{'year': USMALLINT}}
    )
"""

csv_by_year_query = f"""
    SELECT year(date) AS year, count(*) AS csv_rows, sum(trips) AS csv_trips
    FROM (
        SELECT DISTINCT
            "Date" AS date,
            "Hour" AS hour,
            "Origin Station" AS origin,
            "Destination Station" AS dest,
            "Trip Count" AS trips
        FROM {csv}
    )
    GROUP BY year(date)
"""

parquet_by_year_query = f"""
    SELECT year, count(*) AS parquet_rows, sum(trips) AS parquet_trips
    FROM {parquet}
    GROUP BY year
"""

compare_query = """
    SELECT
        c.year,
        c.csv_rows,
        p.parquet_rows,
        c.csv_trips,
        p.parquet_trips,
        c.csv_rows = p.parquet_rows   AS rows_match,
        c.csv_trips = p.parquet_trips AS trips_match
    FROM csv_by_year AS c
    FULL OUTER JOIN parquet_by_year AS p ON c.year = p.year
    ORDER BY c.year
"""

with stage("01.validate"):
    con.execute(f"CREATE OR REPLACE TEMP TABLE csv_by_year AS {csv_by_year_query}")
    con.execute(f"CREATE OR REPLACE TEMP TABLE parquet_by_year AS {parquet_by_year_query}")
    con.execute(f"CREATE OR REPLACE TEMP TABLE compare AS {compare_query}")
    raw_rows, raw_trips = fetch_row(con, f'SELECT count(*), sum("Trip Count") FROM {csv}')

con.sql("SELECT * FROM compare").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[perf] 01.validate: 5.57s | 16.46 cores | peak RAM 8.575 GB | CPU 92.6%


,year,csv_rows,parquet_rows,csv_trips,parquet_trips,rows_match,trips_match
0,2011,9428124,9428124,106423154.0,106423154.0,True,True
1,2012,9769509,9769509,115407813.0,115407813.0,True,True
2,2013,9603756,9603756,116929864.0,116929864.0,True,True
3,2014,9889307,9889307,124011380.0,124011380.0,True,True
4,2015,10072503,10072503,127723465.0,127723465.0,True,True
5,2016,9971582,9971582,128176267.0,128176267.0,True,True
6,2017,9972078,9972078,122849934.0,122849934.0,True,True
7,2018,10227631,10227631,120494260.0,120494260.0,True,True
8,2019,10114285,10114285,118609605.0,118609605.0,True,True
9,2020,5811863,5811863,31426279.0,31426279.0,True,True


In [14]:
mismatched_years = fetch_value(con, """
    SELECT count(*)
    FROM compare
    WHERE rows_match IS NOT TRUE OR trips_match IS NOT TRUE
""")
total_rows, total_trips = fetch_row(con, "SELECT sum(parquet_rows), sum(parquet_trips) FROM compare")

print(f"raw CSV            rows {raw_rows:,}  trips {raw_trips:,}")
print(f"duplicates removed rows {extra_rows:,}  trips {extra_trips:,}")
print(f"parquet            rows {total_rows:,}  trips {total_trips:,}")
print("years that do not match:", mismatched_years)

assert mismatched_years == 0
assert total_rows == raw_rows - extra_rows
assert total_trips == raw_trips - extra_trips

raw CSV            rows 112,536,132  trips 1,194,695,771
duplicates removed rows 16,078  trips 240,983
parquet            rows 112,520,054  trips 1,194,454,788
years that do not match: 0


In [15]:
key_check = con.sql(f"""
    SELECT
        count(*) AS rows,
        count(DISTINCT (date, hour, origin, dest)) AS unique_keys,
        count(*) FILTER (WHERE year(date) <> year) AS wrong_folder_rows
    FROM {parquet}
""").df().iloc[0]

print("rows             :", f"{key_check['rows']:,}")
print("unique keys      :", f"{key_check['unique_keys']:,}")
print("wrong folder rows:", key_check["wrong_folder_rows"])

assert key_check["rows"] == key_check["unique_keys"]
assert key_check["wrong_folder_rows"] == 0

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows             : 112,520,054
unique keys      : 112,520,054
wrong folder rows: 0


In [16]:
con.sql(f"DESCRIBE SELECT * FROM {parquet}").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,hour,UTINYINT,YES,None,None,None
2,origin,VARCHAR,YES,None,None,None
3,dest,VARCHAR,YES,None,None,None
4,trips,USMALLINT,YES,None,None,None
5,year,USMALLINT,YES,None,None,None


In [17]:
first_rows_query = f"""
    SELECT *
    FROM {parquet}
    ORDER BY date, hour, origin, dest
    LIMIT 5
"""

con.sql(first_rows_query).df()

,date,hour,origin,dest,trips,year
0,2011-01-01,0,12TH,12TH,1,2011
1,2011-01-01,0,12TH,16TH,1,2011
2,2011-01-01,0,12TH,24TH,3,2011
3,2011-01-01,0,12TH,ASHB,2,2011
4,2011-01-01,0,12TH,BAYF,5,2011


origin และ dest อ่านกลับมาเป็น VARCHAR เพราะ Parquet ไม่มีชนิด ENUM แต่ในไฟล์เก็บแบบ dictionary อยู่แล้ว
ตอนทำ marts ค่อย cast กลับเป็น ENUM

## ขนาดไฟล์

เทียบขนาด CSV กับ Parquet รายปี

In [ ]:
def folder_size_mb(folder):
    total_bytes = 0
    for file in folder.glob("*.parquet"):
        total_bytes += file.stat().st_size
    return total_bytes / 1e6

csv_mb = {}
for file in RAW.glob("*.csv"):
    year = int(file.stem[-4:])
    csv_mb[year] = file.stat().st_size / 1e6

parquet_mb = {}
for folder in OD.glob("year=*"):
    year = int(folder.name.split("=")[1])
    parquet_mb[year] = folder_size_mb(folder)

sizes = pd.DataFrame({"csv_mb": csv_mb, "parquet_mb": parquet_mb}).sort_index()
sizes["ratio"] = sizes["csv_mb"] / sizes["parquet_mb"]

total_csv_mb = sizes["csv_mb"].sum()
total_parquet_mb = sizes["parquet_mb"].sum()
total_ratio = total_csv_mb / total_parquet_mb

print(f"CSV {total_csv_mb / 1000:.2f} GB -> Parquet {total_parquet_mb:.0f} MB ({total_ratio:.1f}x smaller)")
sizes.round(1)

CSV 2.92 GB -> Parquet 115 MB (25.5x smaller)


,csv_mb,parquet_mb,ratio
2011,245.2,9.7,25.2
2012,254.1,10.1,25.2
2013,249.9,10.0,25.1
2014,257.3,10.3,25.1
2015,262.1,10.5,25.0
2016,259.5,10.4,24.9
2017,259.4,10.4,25.0
2018,265.8,10.6,25.0
2019,262.9,10.6,24.9
2020,150.2,5.9,25.5


In [19]:
csv_bars = go.Bar(name="CSV", x=sizes.index, y=sizes["csv_mb"], marker_color=viz.CONTEXT)
parquet_bars = go.Bar(name="Parquet (zstd)", x=sizes.index, y=sizes["parquet_mb"], marker_color=viz.ACCENT)

fig = go.Figure([csv_bars, parquet_bars])
fig.update_traces(hovertemplate="%{x}: %{y:,.1f} MB<extra>%{fullData.name}</extra>")
fig.update_layout(barmode="group", hovermode="closest")
fig.update_yaxes(title_text="MB")

title = f"Parquet is {total_ratio:.0f}x smaller than CSV"
subtitle = f"File size per year. Total {total_csv_mb / 1000:.2f} GB CSV vs {total_parquet_mb:.0f} MB Parquet"
viz.titled(fig, title, subtitle)
viz.save(fig, "01_csv_vs_parquet")
fig

## Row group

อ่าน metadata ของทุกไฟล์ ดูช่วงวันที่ของแต่ละ row group
ถ้าข้อมูลเรียงจริง row group ถัดไปต้องเริ่มไม่ก่อนวันสุดท้ายของ row group ก่อนหน้า (วันเดียวกันได้ เพราะหนึ่งวันอาจถูกแบ่งอยู่ 2 row group)
แบบนี้ query ที่ filter ช่วงวันที่จะข้าม row group ที่ไม่เกี่ยวได้

In [20]:
def read_row_groups(file, year):
    meta = pq.ParquetFile(file).metadata
    date_column = meta.schema.names.index("date")

    rows = []
    for i in range(meta.num_row_groups):
        row_group = meta.row_group(i)
        date_stats = row_group.column(date_column).statistics
        rows.append({
            "year": year,
            "row_group": i,
            "rows": row_group.num_rows,
            "min_date": date_stats.min,
            "max_date": date_stats.max,
        })
    return rows


def count_overlaps(row_groups):
    overlaps = 0
    for i in range(1, len(row_groups)):
        previous_max = row_groups.iloc[i - 1]["max_date"]
        current_min = row_groups.iloc[i]["min_date"]
        if current_min < previous_max:
            overlaps += 1
    return overlaps

In [21]:
all_row_groups = []
for year in years:
    file = OD / f"year={year}" / "data_0.parquet"
    all_row_groups.extend(read_row_groups(file, year))

row_groups = pd.DataFrame(all_row_groups)

summary = []
for year in years:
    groups_of_year = row_groups[row_groups["year"] == year].reset_index(drop=True)
    summary.append({
        "year": year,
        "row_groups": len(groups_of_year),
        "min_rows": groups_of_year["rows"].min(),
        "max_rows": groups_of_year["rows"].max(),
        "overlaps": count_overlaps(groups_of_year),
    })

row_group_summary = pd.DataFrame(summary).set_index("year")
total_overlaps = row_group_summary["overlaps"].sum()

print("total row groups:", len(row_groups))
print("total overlaps  :", total_overlaps)
assert total_overlaps == 0

row_group_summary

total row groups: 118
total overlaps  : 0


,row_groups,min_rows,max_rows,overlaps
year,,,,
2011,10,414876,1001472,0
2012,10,756261,1001472,0
2013,10,590508,1001472,0
2014,10,876059,1001472,0
2015,11,57783,1001472,0
2016,10,958334,1001472,0
2017,10,958830,1001472,0
2018,11,212911,1001472,0
2019,11,99565,1001472,0


In [22]:
row_groups[row_groups["year"] == 2019].head(10)

,year,row_group,rows,min_date,max_date
82,2019,0,1001472,2019-01-01,2019-02-06
83,2019,1,1001472,2019-02-06,2019-03-14
84,2019,2,1001472,2019-03-14,2019-04-19
85,2019,3,1001472,2019-04-19,2019-05-25
86,2019,4,1001472,2019-05-25,2019-06-30
87,2019,5,1001472,2019-06-30,2019-08-05
88,2019,6,1001472,2019-08-05,2019-09-10
89,2019,7,1001472,2019-09-10,2019-10-15
90,2019,8,1001472,2019-10-15,2019-11-20
91,2019,9,1001472,2019-11-20,2019-12-27


## สรุป

- CSV 13 ไฟล์ 2.92 GB แปลงเป็น Parquet 115 MB เล็กลง 25.5 เท่า
- เขียน Parquet ใช้เวลา 8.1 วินาที (เฉลี่ยใช้ ~13.6 core, RAM สูงสุด 7.3 GB) เช็คความถูกต้องใช้ 5.5 วินาที
- ข้อมูลดิบไม่มีค่าว่าง hour อยู่ในช่วง 0-23 trips อยู่ในช่วง 1-1,826 สถานีครบ 50 สถานี
- เจอแถวซ้ำ 16,078 แถว (240,983 ทริป) เป็นช่วงตี 0-2 ของวันที่ 1 มกราคม ปี 2012-2018 ที่ติดมาในไฟล์ปีก่อนหน้า ค่าตรงกันทุกแถว ตัดออกแล้ว
- หลังตัดเหลือ 112,520,054 แถว 1,194,454,788 ทริป ตรงกับ CSV ทุกปี ไม่มี key ซ้ำ ทุกแถวอยู่ในโฟลเดอร์ปีที่ถูกต้อง
- ข้อมูลเรียงตาม date, hour, origin, dest จริง 118 row group ไม่มีช่วงวันที่ทับกัน แต่ละ row group ประมาณ 1 เดือน
- มีทริปเข้าและออกสถานีเดียวกัน 2.37 ล้านแถว (6.5 ล้านทริป ประมาณ 0.5%) ยังเก็บไว้ ค่อยตัดสินใจใน 02_marts

ต่อไป: 02_marts